# Assignment 2
Authors: Kevin Havis & Aali John-Harry

## Introduction

For this assignment, we selected the Email-Eu-core network, which represents email communication between members of a large European research institution.

In this graph:
- **Nodes** represent employees within the institution.
- **Edges** represent at least one email sent between two employees.
- The dataset includes **department labels** for each employee.

## Data Setup

The dataset consists of two primary text files:

- email-Eu-core.txt
    - contains the directed edge list representing email exchanges.
- email-Eu-core-department-labels.txt
    - contains department labels for each employee.

1. We begin by loading the edge list into a directed graph using NetworkX, since email communication naturally has direction (sender to recipient).
2. We then attach department labels as node attributes, allowing us to incorporate organizational metadata into our analysis and visualization.
3. Because diameter requires a fully connected structure, we extract the largest strongly connected component (SCC) of the graph before computing certain metrics.
    - This ensures that path-based calculations such as diameter and betweenness centrality are meaningful and well-defined.

This setup allows us to move from raw text data to a structured graph model that can be analyzed and visualized using external tools such as Gephi.

In [ ]:
import networkx as nx

# Load as directed graph
G = nx.read_edgelist("email-Eu-core.txt", 
                     create_using=nx.DiGraph(), 
                     nodetype=int)

print("Number of nodes:", G.number_of_nodes())
print("Number of edges:", G.number_of_edges())

In [ ]:
# Add Department Labels
department_labels = {}

with open("email-Eu-core-department-labels.txt") as f:
    for line in f:
        node, dept = line.split()
        department_labels[int(node)] = int(dept)

nx.set_node_attributes(G, department_labels, "department")

In [ ]:
# Largest Strongly Connected Component
largest_scc = max(nx.strongly_connected_components(G), key=len)
G_scc = G.subgraph(largest_scc)

print("Nodes in largest SCC:", G_scc.number_of_nodes())

diameter = nx.diameter(G_scc)
print("Diameter of largest SCC:", diameter)

In [ ]:
betweenness = nx.betweenness_centrality(G_scc)

top_betweenness = sorted(betweenness.items(), key=lambda x: x[1], reverse=True)[:5]

#Top 5 nodes by betweenness
print("Top 5 nodes by betweenness centrality:")
for node, score in top_betweenness:
    print(f"Node {node}: {score:.4f}")

### Network Diameter

* The diameter of the largest strongly connected component is 6.
    * This means that the maximum shortest-path distance between any two employees within the core connected structure is six steps.
    * Simply put, any employee in the largest communication cluster can reach any other employee in at most six email exchanges. 


### Betweenness Centrality

The top five nodes by betweenness centrality are:

160, 86, 5, 62, and 121.

Node 160 has the highest betweenness score (0.095), showing that it likely serves as a major communication bridge between different parts of the organization.

In [ ]:
clustering = nx.average_clustering(G_scc.to_undirected())
print("Average clustering coefficient:", clustering)

### Clustering Coefficient

The average clustering coefficient of the network is approximately 0.43.

This indicates that employees tend to form tightly connected local groups, where many of a person’s contacts also communicate with one another. 